# Radon Analysis

Evaluating mitigation system effectiveness from hourly radon sensor readings.

**Reference levels (pCi/L):**
- **4.0** — EPA action level (mitigate above this)
- **2.7** — WHO recommended action level
- **1.3** — average US indoor radon level

**Timeline:**
- 2026-05-12 — mitigation fan turned on
- 2026-05-14 — sensor moved to current location (readings before this are *not* comparable — sensor was in a lower-radon room)
- 2026-05-18 — improved sealing, vacuum increased from 2.5" to 3.5" WC
- 2026-05-23 — external suction-point adjustments (no internal changes)

Because the sensor moved on **2026-05-14**, all aggregates and statistics below are restricted to that date forward. Pre-move readings appear (shaded) in the raw hourly time-series for context only.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

EPA_LEVEL = 4.0
WHO_LEVEL = 2.7

SENSOR_MOVED = '2026-05-14'  # all analysis restricted to this date forward

# (date, short label, color) — mitigation events in green, sensor event in purple
EVENTS = [
    ('2026-05-12', 'Fan',      'green'),
    ('2026-05-14', 'Sensor →', 'purple'),
    ('2026-05-18', '3.5" WC',  'green'),
    ('2026-05-23', 'Ext.',     'green'),
]

# Phases for comparison (start inclusive, end exclusive; end=None means open-ended)
PHASES = [
    ('P1: fan + 2.5" WC',    '2026-05-14', '2026-05-18'),
    ('P2: 3.5" WC + sealing', '2026-05-18', '2026-05-23'),
    ('P3: + ext. tweaks',     '2026-05-23', None),
]

## Load data

In [2]:
df = pd.read_csv(
    '03-01-2026_06-02-2026-export.csv',
    parse_dates=['Date'],
    date_format='%m/%d/%Y %H:%M',
)
df = df.rename(columns={'Date': 'ts', 'Radon Level': 'radon'}).set_index('ts').sort_index()

df_post = df.loc[SENSOR_MOVED:].copy()  # comparable readings only

print(f'Full dataset:     {len(df):>5} rows  |  {df.index.min()} → {df.index.max()}')
print(f'Post-sensor-move: {len(df_post):>5} rows  |  {df_post.index.min()} → {df_post.index.max()}')
df.head()

Full dataset:       732 rows  |  2026-04-25 19:24:00 → 2026-06-02 20:31:00
Post-sensor-move:   346 rows  |  2026-05-14 00:25:00 → 2026-06-02 20:31:00


,radon
ts,
2026-04-25 19:24:00,1.2
2026-04-25 20:24:00,0.9
2026-04-25 21:24:00,0.4
2026-04-25 22:24:00,0.5
2026-04-25 23:24:00,0.4


## Helper: chart annotations

Uses `add_shape` + `add_annotation` directly rather than `add_hline/vline/vrect`'s `annotation_text=` parameter, to sidestep a plotly bug that trips on pandas 3.0 Timestamp arithmetic.

In [3]:
EVENTS_POST = [(d, l, c) for d, l, c in EVENTS if pd.Timestamp(d) >= pd.Timestamp(SENSOR_MOVED)]


def annotate(fig, *, levels=True, events=None, shade_pre_move=False, height=600):
    """Add EPA/WHO horizontal lines, event vertical lines, and optional pre-move shading."""
    fig.update_layout(height=height)
    if levels:
        for y, color, dash, name in [(EPA_LEVEL, 'red', 'dash', 'EPA'),
                                     (WHO_LEVEL, 'orange', 'dot', 'WHO')]:
            fig.add_shape(type='line', xref='x domain', yref='y',
                          x0=0, x1=1, y0=y, y1=y,
                          line=dict(color=color, dash=dash, width=1))
            fig.add_annotation(xref='x domain', yref='y',
                               x=1, y=y, text=f'{name} {y}', showarrow=False,
                               xanchor='right', yanchor='bottom',
                               font=dict(size=10, color=color))
    if events:
        for i, (date, label, color) in enumerate(events):
            x = pd.Timestamp(date)
            fig.add_shape(type='line', xref='x', yref='y domain',
                          x0=x, x1=x, y0=0, y1=1,
                          line=dict(color=color, dash='dash', width=1.5))
            on_top = (i % 2 == 0)
            fig.add_annotation(xref='x', yref='y domain',
                               x=x, y=1.02 if on_top else -0.04,
                               text=label, showarrow=False,
                               xanchor='center',
                               yanchor='bottom' if on_top else 'top',
                               font=dict(size=10, color=color))
    if shade_pre_move:
        fig.add_shape(type='rect', xref='x', yref='y domain',
                      x0=df.index.min(), x1=pd.Timestamp(SENSOR_MOVED),
                      y0=0, y1=1,
                      fillcolor='gray', opacity=0.18, line_width=0, layer='below')
        fig.add_annotation(xref='x', yref='y domain',
                           x=df.index.min(), y=0.97,
                           text='not comparable (sensor in other room)',
                           showarrow=False,
                           xanchor='left', yanchor='top',
                           font=dict(size=10, color='gray'))
    return fig

## Raw hourly time series (full dataset)

Shaded region = sensor was in a different room; values are not comparable to post-move readings.

In [4]:
fig = px.line(df, y='radon', title='Hourly radon (pCi/L) — full dataset',
              labels={'radon': 'pCi/L', 'ts': ''})
fig.update_traces(line=dict(width=1))
annotate(fig, events=EVENTS, shade_pre_move=True)
fig.show()

---

# Post-sensor-move analysis

Everything below uses **`df_post`** — readings from 2026-05-14 onward only.

In [5]:
df_post['radon'].describe().round(2)

count    346.00
mean       1.24
std        1.54
min        0.20
25%        0.60
50%        0.70
75%        1.20
max       15.00
Name: radon, dtype: float64

## Daily aggregates

In [6]:
daily = df_post['radon'].resample('D').agg(['mean', 'min', 'max', 'count'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily['max'], name='daily max',
                         line=dict(color='lightgray'), mode='lines'))
fig.add_trace(go.Scatter(x=daily.index, y=daily['min'], name='daily min',
                         line=dict(color='lightgray'), mode='lines', fill='tonexty'))
fig.add_trace(go.Scatter(x=daily.index, y=daily['mean'], name='daily mean',
                         line=dict(color='steelblue', width=2.5)))
fig.update_layout(title='Daily radon: mean (line) with min–max band', yaxis_title='pCi/L', xaxis_title='')
annotate(fig, events=EVENTS_POST)
fig.show()

## Moving averages

The 24-hour window smooths out the diurnal cycle; the 7-day window smooths out weekly weather/usage patterns.

Note: with only ~3 weeks of post-move data, the 7-day MA doesn't settle until around May 21.

In [7]:
ma = pd.DataFrame({
    'hourly': df_post['radon'],
    '24h MA': df_post['radon'].rolling('24h').mean(),
    '7d MA':  df_post['radon'].rolling('7D').mean(),
})

fig = px.line(ma, title='Radon with moving averages',
              labels={'value': 'pCi/L', 'ts': '', 'variable': ''})
fig.update_traces(selector=dict(name='hourly'), line=dict(color='lightgray', width=1))
fig.update_traces(selector=dict(name='24h MA'), line=dict(color='steelblue', width=2))
fig.update_traces(selector=dict(name='7d MA'),  line=dict(color='crimson', width=2.5))
annotate(fig, events=EVENTS_POST)
fig.show()

## Hour-of-day pattern

Radon often peaks overnight (less air exchange) and dips during the day. If mitigation is effective and running 24/7, you'd expect this pattern to flatten.

In [8]:
by_hour = df_post.copy()
by_hour['hour'] = by_hour.index.hour

fig = px.box(by_hour, x='hour', y='radon', points=False,
             title='Radon distribution by hour of day (post-sensor-move)',
             labels={'radon': 'pCi/L', 'hour': 'hour of day'})
annotate(fig)
fig.show()

## Day-of-week pattern

In [9]:
by_dow = df_post.copy()
by_dow['dow'] = by_dow.index.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = px.box(by_dow, x='dow', y='radon', points=False,
             category_orders={'dow': dow_order},
             title='Radon distribution by day of week (post-sensor-move)',
             labels={'radon': 'pCi/L', 'dow': ''})
annotate(fig)
fig.show()

## Threshold exceedance (post-sensor-move)

In [10]:
n = len(df_post)
above_epa = (df_post['radon'] >= EPA_LEVEL).sum()
above_who = (df_post['radon'] >= WHO_LEVEL).sum()
idxmax = df_post['radon'].idxmax()

print(f'Hourly readings:    {n}')
print(f'>= EPA {EPA_LEVEL}:        {above_epa:>4}  ({above_epa / n:.1%})')
print(f'>= WHO {WHO_LEVEL}:        {above_who:>4}  ({above_who / n:.1%})')
print(f'Mean:               {df_post["radon"].mean():.2f}')
print(f'Median:             {df_post["radon"].median():.2f}')
print(f'Max:                {df_post["radon"].max():.2f}   on {idxmax}')

Hourly readings:    346
>= EPA 4.0:          17  (4.9%)
>= WHO 2.7:          34  (9.8%)
Mean:               1.24
Median:             0.70
Max:                15.00   on 2026-05-16 22:29:00


## Phase comparison

Three mitigation phases bounded by the May 18 and May 23 events. Phases are short (especially P1 at ~4 days), so treat differences as suggestive rather than conclusive.

In [11]:
def phase_stats(s):
    if len(s) == 0:
        return pd.Series({'days': 0, 'n': 0, 'mean': float('nan'), 'median': float('nan'),
                          'p90': float('nan'), 'max': float('nan'),
                          '%_above_EPA': 0.0, '%_above_WHO': 0.0})
    span_days = (s.index.max() - s.index.min()).total_seconds() / 86400
    return pd.Series({
        'days':         round(span_days, 1),
        'n':            len(s),
        'mean':         round(s.mean(), 2),
        'median':       round(s.median(), 2),
        'p90':          round(s.quantile(0.9), 2),
        'max':          round(s.max(), 2),
        '%_above_EPA':  round((s >= EPA_LEVEL).mean() * 100, 1),
        '%_above_WHO':  round((s >= WHO_LEVEL).mean() * 100, 1),
    })

phase_series = {}
for name, start, end in PHASES:
    start_ts = pd.Timestamp(start)
    end_ts   = pd.Timestamp(end) if end else (df.index.max() + pd.Timedelta(seconds=1))
    mask = (df.index >= start_ts) & (df.index < end_ts)
    phase_series[name] = df.loc[mask, 'radon']

summary = pd.DataFrame({name: phase_stats(s) for name, s in phase_series.items()}).T
summary

,days,n,mean,median,p90,max,%_above_EPA,%_above_WHO
"P1: fan + 2.5"" WC",4.0,70.0,2.16,1.3,4.76,15.0,14.3,25.7
"P2: 3.5"" WC + sealing",5.0,89.0,0.88,0.8,1.42,3.8,0.0,2.2
P3: + ext. tweaks,10.4,187.0,1.07,0.7,2.34,10.3,3.7,7.5


In [12]:
phase_long = pd.concat([
    s.to_frame('radon').assign(phase=name)
    for name, s in phase_series.items()
]).reset_index()

fig = px.box(phase_long, x='phase', y='radon', points='outliers',
             title='Radon distribution by mitigation phase',
             labels={'radon': 'pCi/L', 'phase': ''})
annotate(fig)
fig.show()